## 3D SIM reconstruction template

This is a work in progress template for running 3D SIM reconstruction based on the Janelia Python and c SIM code written by David Hoffman and Lin Shao

## 1.  Define code paths
Currently we hard code these and they need to be modified to run on different machines.  In the future we may move to a more intelligent approach like always having code exist beside the notebooks and using relative imports.  

In [55]:

%pylab inline
import mrc
import tifffile as tif
import os
import glob
%load_ext autoreload
%autoreload 
import shutil

# NOTE: BN the below code is from the legacy notebooks.  I am leaving it here for now, as it may be needed on some machines. 
if 'C:\\Users\\Cryo SIM-PALM\\Documents\\GitHub' in sys.path:
    sys.path.remove('C:\\Users\\Cryo SIM-PALM\\Documents\\GitHub')
else:
    pass

computer = 'default'

import sys

if computer == 'default':
    sys.path.insert(1, r'Y:\Cryo_data2\Data Processing Notebooks')
    sys.path.insert(1, r'C:\Users\Cryo SIM-PALM\code\simrecon\scripts')
    sys.path.insert(1, r'C:\Users\Cryo SIM-PALM\code\simrecon\scripts\Scripts')
elif computer == 'bnort':
    sys.path.insert(1, r'C:\Users\bnort\work\Janelia\code\\simrecon\scripts\Scripts')
    sys.path.insert(1, r'C:\Users\bnort\work\Janelia\code\\simrecon\scripts')
else:
    pass

import dphutils 
from simrecon_utils import simrecon, split_process_recombine

# import dask
import dask
from dask.diagnostics import ProgressBar

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Setup home directory and OTF directory:

Right now we leave in paths for the current test machines but in the future may move to a more intelligent approach (for example user chooses paths with dialog box, paths stored in configuration)

Home should start from {Hesslab(\\prfs.hhmi.org)} e.g. home = r'Y:\Cryo_data2\ORCA_data\3D SIM Cells'

#### Additonal legacy notes (BN: these notes were in the original notebook I got) I'm leaving them in for now in case we need them for troubleshhoting

There was an old note "OTF folder should be placed inside Data processing notebooks"

(BN I don't think this has to be the case, because there are many notebook that define the full OTF path)

OTF Folder directory should be e.g.:   Y:\Cryo_data2\Data Processing Notebooks\SIM PSFs OTFs

Raw data from V-SIM data acquisition should be placed inside a dated folder here:   'Y:\Cryo_data2\ORCA_data\3D SIM Cells' 

with data file directory structure e.g.: 'Y:\Cryo_data2\ORCA_data\3D SIM Cells\20240322\488 nm 5 phases 0.81 NA Linear SIM_cam1_0.mrc'

In [67]:
if computer == 'default': 
    home = r"Y:\data..."
    otf_path = r"Y:\data..."
    #'Y:\Seyforth\Data For Brian\Cryo-SIM Scope #1 Data (Ground truth baseline)\Ground truth OTFsSystem #1'
elif computer == 'bnort':
    #home = r'D:\Janelia\Data 2024-06-06\Wiener, gammaApo and SupressR parameter testing\488nm comparison Brian'
    #home = r'D:\Janelia\Data 2024-10-02\560cm cell 4 _20240627_124604'
    home = r'W:\data' 
    
    #otf_path = r'D:\Janelia\Data 2024-06-06\Wiener, gammaApo and SupressR parameter testing\OTF\BEAD 2 - NON-AR 1.2W 25ms retake_20240503_170242 BEST PSF!!\computed_OTF_folder'
    #otf_path = r'D:\Janelia\Data 2024-06-03\PSF-OTF used (Davids set of 4 wavelengths)\201909_19-20_best'
    #otf_path = r'C:\Users\bnort\work\Janelia\ims\OTF_folder'
OTFpath = os.path.join(otf_path,"*{}*.mrc")
OTFs = {wl : [path for path in glob.iglob(OTFpath.format(wl))] for wl in (560, 532, 488, 642)}
print(home)
OTFs

Z:\forJamesS\LID817_ROI4\LID817_ROI4_642


{560: ['Y:\\Seyforth\\Data For Brian\\Brians OTF\\560 201909_19-20_best.mrc'],
 532: ['Y:\\Seyforth\\Data For Brian\\Brians OTF\\532 OTF Bead 3_20190920_154920.mrc'],
 488: ['Y:\\Seyforth\\Data For Brian\\Brians OTF\\488 nmLinOTF0_mask.mrc'],
 642: ['Y:\\Seyforth\\Data For Brian\\Brians OTF\\642 20240611_125236_best.mrc']}

## New Full frame Simrecon code splits data into smaler z chunks to avoid failed processing if data is too large for simrecon C# to process



In [68]:
import concurrent.futures


def _write_and_reconstruct_chunk(c, z_start, z_end, nz_total, frames_per_z,
                                  data, oldmrc, work_dir, base_name,
                                  sim_kwargs, pad_planes):
    pad_lo = min(pad_planes, z_start)
    pad_hi = min(pad_planes, nz_total - z_end)
    z_lo_padded = z_start - pad_lo
    z_hi_padded = z_end + pad_hi
    frame_lo = z_lo_padded * frames_per_z
    frame_hi = z_hi_padded * frames_per_z

    chunk_data = data[frame_lo:frame_hi]

    chunk_raw_path = os.path.join(work_dir, f"_zchunk{c}_{base_name}")
    chunk_out_path = chunk_raw_path.replace('.mrc', '_proc.mrc')

    print(f"    Chunk {c}: writing raw z[{z_lo_padded}:{z_hi_padded}] "
          f"(core z[{z_start}:{z_end}], pad_lo={pad_lo}, pad_hi={pad_hi})")
    Mrc.save(chunk_data, chunk_raw_path, hdr=oldmrc.hdr, ifExists='overwrite')

    chunk_kwargs = sim_kwargs.copy()
    chunk_kwargs['input_file'] = chunk_raw_path
    chunk_kwargs['output_file'] = chunk_out_path

    print(f"    Chunk {c}: reconstructing...")
    result = simrecon(**chunk_kwargs)
    assert os.path.exists(chunk_out_path), (
        f"Chunk {c} reconstruction did not produce an output file. "
        f"simrecon() returned: {result}"
    )
    out_size = os.path.getsize(chunk_out_path)
    print(f"    Chunk {c}: output size = {out_size / 1e9:.2f} GB")

    return {
        'raw_path': chunk_raw_path,
        'out_path': chunk_out_path,
        'pad_lo': pad_lo,
        'pad_hi': pad_hi,
        'core_z': z_end - z_start,
    }


def split_z_process_recombine(fullpath, sim_kwargs, n_chunks=2, pad_planes=8,
                               work_dir=None, cleanup=True):
    """
    Split a raw SIM stack into n_chunks pieces along Z (with padding for
    axial OTF context), reconstruct each piece with simrecon() -- running
    all chunks CONCURRENTLY via threads -- crop off the padding, and
    concatenate results back into one full-Z output.

    Assumes 'fastSIM' raw data ordering: frames are (z, direction, phase),
    i.e. each z-position is a contiguous block of ndirs*nphases frames.
    """
    ndirs = sim_kwargs['ndirs']
    nphases = sim_kwargs['nphases']
    frames_per_z = ndirs * nphases

    oldmrc = Mrc.Mrc(fullpath)
    data = oldmrc.data  # shape (total_frames, ny, nx)
    total_frames, ny, nx = data.shape

    assert total_frames % frames_per_z == 0, (
        f"{total_frames} raw frames is not divisible by ndirs*nphases="
        f"{frames_per_z}. Check ordering assumptions before proceeding."
    )
    nz_total = total_frames // frames_per_z
    assert nz_total % n_chunks == 0, (
        f"{nz_total} z-planes is not evenly divisible into {n_chunks} "
        f"chunks. Pick a divisor of {nz_total} for n_chunks."
    )
    core_z = nz_total // n_chunks

    work_dir = work_dir or os.path.dirname(fullpath)
    base_name = os.path.basename(fullpath)

    chunk_args = []
    for c in range(n_chunks):
        z_start = c * core_z
        z_end = z_start + core_z
        chunk_args.append((c, z_start, z_end))

    # Run all chunks concurrently. This works even though Python threads
    # share the GIL, because each chunk's real CPU work happens inside a
    # separate sirecon.exe OS process launched via subprocess -- the GIL
    # is released while Python waits on that subprocess.
    chunk_meta = [None] * n_chunks
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_chunks) as executor:
        futures = {
            executor.submit(
                _write_and_reconstruct_chunk, c, z_start, z_end, nz_total,
                frames_per_z, data, oldmrc, work_dir, base_name, sim_kwargs,
                pad_planes
            ): c
            for c, z_start, z_end in chunk_args
        }
        for future in concurrent.futures.as_completed(futures):
            c = futures[future]
            chunk_meta[c] = future.result()

    cropped_arrays = []
    last_hdr = None
    for m_info in chunk_meta:
        m = Mrc.Mrc(m_info['out_path'])
        arr = m.data
        last_hdr = m.hdr
        z0 = m_info['pad_lo']
        z1 = arr.shape[0] - m_info['pad_hi']
        got = z1 - z0
        assert got == m_info['core_z'], (
            f"Expected {m_info['core_z']} core planes after cropping, got "
            f"{got}. Output z-count may not match input 1:1 (check zzoom)."
        )
        cropped_arrays.append(np.array(arr[z0:z1]))

    final = np.concatenate(cropped_arrays, axis=0)
    final_path = sim_kwargs['output_file']
    print(f"    Saving final stitched reconstruction: {final_path} "
          f"shape={final.shape}")
    Mrc.save(final, final_path, hdr=last_hdr, ifExists='overwrite')

    if cleanup:
        for m_info in chunk_meta:
            for p in (m_info['raw_path'], m_info['out_path']):
                try:
                    os.remove(p)
                except OSError:
                    pass

    return final_path

## Run this code to process your data using split z chunk method

In [69]:
import os
import glob
import numpy as np
import dask
from dask.diagnostics import ProgressBar
import mrc as Mrc

from simrecon_utils import simrecon
# NOTE: return_wl_otfs must already be defined earlier in this notebook
# (the function that maps a raw file path -> (wl, otfs)). This cell does
# not redefine it.


# ===========================================================================
# CONFIG -- edit these for your run
# ===========================================================================

nofilteroverlaps = True 
gammaApo = 0.7
suppressR = 15.0
user_text = ""

SAFE_OUTPUT_BYTES = int(1.8e9)

Z_SPLIT_N_CHUNKS = 2
Z_SPLIT_PAD_PLANES = 8

base_kwargs = dict(
    nphases=5,
    ndirs=3,
    angle0=1.29,
    negDangle=True,
    na=0.85,
    nimm=1.0,
    zoomfact=2.0,
    background=100.0,
    fastSIM=True,
    otfRA=True,
    dampenOrder0=True,
    k0searchall=True,
    equalizez=True,
    preciseapo=True,
)


# ===========================================================================
# Wavelength -> reconstruction parameter lookups
# ===========================================================================

def standard_wiener(wl):
    return {488: 0.001, 532: 0.001, 560: 0.002, 642: 0.002}.get(wl, 0.001)


def nofilteroverlaps_params(wl):
    table = {
        488: (0.0001, [1.0, 0.1]),
        532: (0.0002, [1.0, 0.2]),
        560: (0.0002, [1.0, 0.2]),
        642: (0.0002, [1.0, 0.2]),
    }
    return table.get(wl, (0.0002, [1.0, 0.2]))


# ===========================================================================
# Output size estimation + routing (int32 overflow FIXED here)
# ===========================================================================

def estimate_output_bytes(raw_path, zoomfact, ndirs, nphases, out_dtype_bytes=4):
    m = Mrc.Mrc(raw_path)
    nx, ny, total_frames = (int(v) for v in m.hdr.Num)  # cast away numpy int32
    frames_per_z = ndirs * nphases
    nz = total_frames // frames_per_z
    out_nx = int(round(nx * zoomfact))
    out_ny = int(round(ny * zoomfact))
    return out_nx * out_ny * nz * out_dtype_bytes


def process(sim_kwargs):
    est_bytes = estimate_output_bytes(
        sim_kwargs["input_file"],
        sim_kwargs.get("zoomfact", 1),
        sim_kwargs["ndirs"],
        sim_kwargs["nphases"],
    )

    if est_bytes > SAFE_OUTPUT_BYTES:
        print(
            f"  -> estimated output {est_bytes / 1e9:.2f} GB exceeds "
            f"{SAFE_OUTPUT_BYTES / 1e9:.2f} GB safe limit, using z-split"
        )
        return dask.delayed(split_z_process_recombine)(
            fullpath=sim_kwargs["input_file"],
            sim_kwargs=sim_kwargs,
            n_chunks=Z_SPLIT_N_CHUNKS,
            pad_planes=Z_SPLIT_PAD_PLANES,
        )
    else:
        print(f"  -> estimated output {est_bytes / 1e9:.2f} GB, direct simrecon()")
        return dask.delayed(simrecon)(**sim_kwargs)


# ===========================================================================
# Main queueing logic
# ===========================================================================

def find_already_processed(home):
    return set(glob.iglob(os.path.join(home, "**", "*proc*.mrc"), recursive=True))


def queue_all(home):
    to_process = []
    done_already = find_already_processed(home)

    if done_already:
        print(
            "Datasets already reconstructed:\n"
            + "\n".join(sorted({os.path.dirname(p) for p in done_already}))
        )

    raw_files = sorted(
        glob.iglob(os.path.join(home, "**", "*SIM*cam1_1*.mrc"), recursive=True)
    )
    raw_files = [f for f in raw_files if "_zchunk" not in os.path.basename(f)]
    step = 0
    for raw in raw_files:
        if "proc" in os.path.basename(raw):
            continue

        wl, otfs = return_wl_otfs(raw)

        kwargs = base_kwargs.copy()
        if nofilteroverlaps:
            wiener, force_modamp = nofilteroverlaps_params(wl)
            kwargs.update(nofilteroverlaps=True, forcemodamp=force_modamp)
            tag = f"_NoFoverlaps_Fmod_{force_modamp}"
        else:
            wiener = standard_wiener(wl)
            kwargs.update(nofilteroverlaps=False)
            tag = ""

        kwargs.update(nthreads=7, gammaApo=gammaApo, suppressR=suppressR, wiener=wiener)
        kwargs["ls"] = (wl / 1000) / (2 * 0.775)

        for otf in otfs:
            step += 1
            otf_name = os.path.splitext(os.path.basename(otf))[0]
            user_text_process = f"{user_text}{tag}_w_{wiener}"

            sim_kwargs = kwargs.copy()
            sim_kwargs["input_file"] = raw
            sim_kwargs["otf_file"] = otf

            output_path = raw.replace(
                ".mrc", f"_proc_{otf_name}_{user_text_process}.mrc"
            )
            sim_kwargs["output_file"] = output_path

            if len(output_path) > 255:
                print(
                    f"[{step}] SKIPPING - path too long (>255 chars): {output_path}\n"
                    "  Fix: shorten the filename or move data to a shallower "
                    "directory."
                )
                continue

            if output_path in done_already:
                print(f"[{step}] Already processed, skipping: {output_path}")
                continue

            print(f"[{step}] Queuing: {output_path}")
            to_process.append(process(sim_kwargs))

    return to_process


# ===========================================================================
# RUN
# ===========================================================================

to_process = queue_all(home)
print(f"\n{len(to_process)} job(s) queued.\n")

with ProgressBar():
    results = dask.compute(*to_process)

Datasets already reconstructed:
Z:\forJamesS\LID817_ROI4\LID817_ROI4_642
[1] Queuing: Z:\forJamesS\LID817_ROI4\LID817_ROI4_642\642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc_642 20240611_125236_best__NoFoverlaps_Fmod_[1.0, 0.2]_w_0.0002.mrc
  -> estimated output 0.54 GB, direct simrecon()

1 job(s) queued.

[########################################] | 100% Completed | 154.59 s


In [9]:
# David Solecki's suggestion: wiener=0.007 gammaApo=0.7 and suppressR=15
# original values by D.Hoffman: wiener=0.001 gammaApo=0.1 and suppressR=1.5

base_kwargs = dict(
                    nphases=5,
                    ndirs=3,
                    angle0= 1.29,
                    negDangle=True,              # James made False to try experiment
                    na= 0.85,
                    nimm= 1.0,
                    zoomfact= 2.0, 
                    background= 100.0,           # james experiment was 100.0
                    wiener= 0.007,
                    fastSIM=True,
                    otfRA= True,
                    dampenOrder0=True,
                    k0searchall=True,
                    equalizez=True,
                    preciseapo=True,
                    gammaApo=0.7,
                    suppressR=15.0
                )

def return_wl_otfs(path):
    if "488 Exc 532 Em" in path:
        wl = 532
    elif "488 Exc 642 Em" in path:
        wl = 642
    elif "532 Exc 561 Em" in path:
        wl = 561
    elif "560 nm" in path:
        wl = 560
    elif "488 nm" in path:
        wl = 488
    elif "532 nm" in path:
        wl = 532
    elif "642 nm" in path:
        wl = 642

    else:
        raise RuntimeError("no matching filename wavelength found, fix directory or filename or code")
    return wl, OTFs[wl]

In [13]:
#parameter_sets_excitation_emission

#532nm excitation 650nm emission
dict_532nm_650nm = dict()
dict_532nm_650nm['wiener'] = .001
dict_532nm_650nm['forcemodamp'] = [0.8,0.25]
dict_532nm_650nm['otfamp'] = [1,1]
dict_532nm_650nm['nofilteroverlaps'] = True

#532nm excitation 650nm emission
dict_561nm_650nm = dict()
dict_561nm_650nm['wiener'] = .001
dict_561nm_650nm['forcemodamp'] = [0.8,0.25]
dict_561nm_650nm['otfamp'] = [1,1]
dict_561nm_650nm['nofilteroverlaps'] = True

#640nm excitation 650nm emission # default 642nm parameters
dict_642nm_650nm = dict()
dict_642nm_650nm['wiener'] = .0005
dict_642nm_650nm['forcemodamp'] = [0.8,0.25]
dict_642nm_650nm['otfamp'] = [1,1]
dict_642nm_650nm['nofilteroverlaps'] = True

#488nm excitation 515nm emission # default 488nm parameters
dict_488nm_515nm = dict()
dict_488nm_515nm['wiener'] = .0001
dict_488nm_515nm['forcemodamp'] = [0.8,0.15]
dict_488nm_515nm['otfamp'] = [1,1]
dict_488nm_515nm['nofilteroverlaps'] = True

#561nm excitation 615nm emission # default 561nm parameters
dict_488nm_515nm = dict()
dict_488nm_515nm['wiener'] = .00025
dict_488nm_515nm['forcemodamp'] = [0.8,0.25]
dict_488nm_515nm['otfamp'] = [1,1]
dict_488nm_515nm['nofilteroverlaps'] = True

# Tiled reconstruction method below


## Check if data has been procssed already 

In [53]:
for raw in glob.iglob(os.path.join(home, "**", "*SIM_cam1_1*.mrc"), recursive=True):
    print(os.path.split(raw)[-1])  # See what the full paths actually look like
    #print(raw)  # or process the file
    done_already = set(glob.iglob(home + "/**/*proc*.mrc", recursive = True))
    done_already;
    #done_already = set()     # do this if want to re-process
    #print(done_already)


488 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1.mrc
488 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc_488 nmLinOTF0_mask__NoFoverlaps_Fmod_[1.0, 0.1]_w_0.0001.mrc
_zchunk0_488 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc.mrc
_zchunk1_488 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc.mrc


# Select tile parmeters for tile reconstruction

In [54]:
background =100.0
SuppressR = 1.0
base_kwargs = dict(
                    nphases=5,
                    ndirs=3,
                    angle0= 1.29,
                    negDangle=True,              # James made False to try experiment
                    na= 0.85,
                    nimm= 1.0,
                    zoomfact= 2.0, 
                    background= background,           # james experiment was 100.0
                    wiener= 0.007,
                    fastSIM=True,
                    otfRA= True,
                    dampenOrder0=True,
                    k0searchall=False,
                    equalizez=True,
                    preciseapo=True,
                    gammaApo=0.7,
                    suppressR=SuppressR
                )


In [69]:



# # # Tile size Parameters (units: pixels) # # #
tile_size = 128   #square tile size
tile_overlap = int(tile_size/2)        # square tile overlap [usually = tile_size/2]

 # # perform SIM recon wih No Filter overlaps # # #
    
nofilteroverlaps =  True

# # # TILE Parameter settings # # # 
if tile_size == 64:
    # SIM RECON PARAMS
    gammaApo = 0.5
    suppressR = 2.0
    #force_modamp = [0.8,0.1]  # [0.8,0.1] for 488nm,[0.8,0.25] for 532nm, [0.8,0.4] for 561nm
elif tile_size == 32:
    # SIM RECON PARAMS
    gammaApo = 0.05
    suppressR = 1
elif tile_size == 16:
    # SIM RECON PARAMS
    gammaApo = 0.5
    suppressR = 0.5
elif tile_size == 1024:
    # SIM RECON PARAMS
    gammaApo = 0.5
    suppressR = 2.0
elif tile_size == 128:
    # SIM RECON PARAMS
    gammaApo = 0.5
    suppressR = 6.0

user_text=''


# # # Constrained tile filter method # # #
filter_tiles = False
    
#user_text += user_text1 + user_text2 + user_text3 + user_text4 # comment out if your string/driectory path too long


keep_order2 = False

#update dataset processd, done_already list
done_already = []

import os
import glob
import mrcfile

# Be specific about the subfolder

# Ensure this is defined before the loop
current_tile_size = 64 

import os
import glob
import mrc

# Checking dimensions are compatible with Tiling reconstcution 

for raw in glob.iglob(os.path.join(home, "**", "*SIM*cam1_1*.mrc"), recursive=True):
    print(os.path.split(raw)[-1])  # See what the full paths actually look like
    #print(raw)  # or process the file
    done_already = set(glob.iglob(home + "/**/*proc*.mrc", recursive = True))
    done_already;
    #done_already = set()     # do this if want to re-process
    #print(done_already)
    filename = os.path.split(raw)[1]

    # 1. EXCEPTION GATEKEEPER: Kill script if 'proc' is found
    if "proc" not in os.path.split(raw)[1]: 

        try:
            # 2. Get dimensions using your working method
            mymrc = mrc.Mrc(raw) 
            nz, ny, nx = mymrc.data.shape

            # 3. MATH GATEKEEPER: Find the largest 2^n divisor
            # Checks 2^10 (1024) down to 2^5 (32)
            selected_tile = 0
            for n in range(10, 3, -1): 
                tile_attempt = 2**n
                if nx % tile_attempt == 0 and ny % tile_attempt == 0:
                    selected_tile = tile_attempt
                    break

            # 4. VALIDATION DECISION
            if selected_tile == 0:
                print(f"!!! REJECTED: {filename} ({nx}x{ny}) is not tileable by 2^n (32-1024).")
                continue 

            print(f"Processing {filename} ({nx}x{ny}) using Tiling recon is possible with Selected Tile: {tile_size}")

        except Exception as e:
            print(f"!!! ERROR reading {filename}: {e}")
            continue

    # --- PROCEED WITH RECONSTRUCTION ---
    # current_tile_size is now set to the best 2^n fit for this specific file

# print("Datasets alread reconstructed: \n" + "\n".join(os.path.split(i)[0] for i in done_already))
    


# for raw in glob.iglob(home + "/**/*SIM_cam1_1*.mrc", recursive = True):
    
    
    
    user_text_process = user_text
    wl, otfs = return_wl_otfs(raw)
    
    
    if filter_tiles:
        
    
        tile_limits = {}
   
        if wl == 488:
            tile_limits['spacing_min'] = 0.31   # magnitude vector inverse to line spacing value here; see simrecon text file output
            tile_limits['spacing_max'] = 0.32
            tile_limits['spacing_default']=0.361 # SIM angle
            tile_limits['angle_min'] = 1.2
            tile_limits['angle_max'] = 1.3
            tile_limits['angle_default'] = 1.29
            tile_limits['amp1_min'] = 0.0   # contrast of order 1
            tile_limits['amp1_max'] = 1.0
            tile_limits['amp1_default'] = 1.0
            tile_limits['amp2_min'] = 0.0  # contrast of order 2
            tile_limits['amp2_max'] = 1.0
            tile_limits['amp2_default'] = 1.0

        elif wl == 560:
            tile_limits['spacing_min'] = 0.355   # magnitude vector inverse to line spacing value here; see simrecon text file output
            tile_limits['spacing_max'] = 0.365
            tile_limits['spacing_default']=0.361 # SIM angle
            tile_limits['angle_min'] = 1.3
            tile_limits['angle_max'] = 1.4
            tile_limits['angle_default'] = 1.36
            tile_limits['amp1_min'] = 0.98   # contrast of order 1
            tile_limits['amp1_max'] = 1.02
            tile_limits['amp1_default'] = 1.0
            tile_limits['amp2_min'] = 0.98  # contrast of order 2
            tile_limits['amp2_max'] = 1.02
            tile_limits['amp2_default'] = 1.0

        elif wl == 642:
            tile_limits['spacing_min'] = 0.408  # magnitude vector inverse to line spacing value here; see simrecon text file output
            tile_limits['spacing_max'] = 0.416
            tile_limits['spacing_default']=0.412 # SIM angle
            tile_limits['angle_min'] = 1.2
            tile_limits['angle_max'] = 1.6
            tile_limits['angle_default'] = 1.36
            tile_limits['amp1_min'] = 0.98   # contrast of order 1
            tile_limits['amp1_max'] = 1.02
            tile_limits['amp1_default'] = 1.0
            tile_limits['amp2_min'] = 0.98  # contrast of order 2
            tile_limits['amp2_max'] = 1.02
            tile_limits['amp2_default'] = 1.0

        elif wl == 532:
            tile_limits['spacing_min'] = 0.336  # magnitude vector inverse to line spacing value here; see simrecon text file output
            tile_limits['spacing_max'] = 0.348
            tile_limits['spacing_default']=0.412 # SIM angle
            tile_limits['angle_min'] = 1.2
            tile_limits['angle_max'] = 1.6
            tile_limits['angle_default'] = 1.36
            tile_limits['amp1_min'] = 0.98   # contrast of order 1
            tile_limits['amp1_max'] = 1.02
            tile_limits['amp1_default'] = 1.0
            tile_limits['amp2_min'] = 0.98  # contrast of order 2
            tile_limits['amp2_max'] = 1.02
            tile_limits['amp2_default'] = 1.0
    
        user_text1 = 'mag_' + str(tile_limits['spacing_min'])  +"_" + str(tile_limits['spacing_max'])
        user_text2 = 'ang_' + str(tile_limits['angle_min'])  +"_" + str(tile_limits['angle_max'])
        user_text3 = 'A1_' + str(tile_limits['amp1_min'])  +"_" + str(tile_limits['amp1_max'])
        user_text4 = 'A2_' + str(tile_limits['amp2_min'])  +"_" + str(tile_limits['amp2_max'])
#         print('\n' + '\n' "TILING PARAMETERS: " + '_spacing_' + str(tile_limits['spacing_min'])  
#         +"_to_" + str(tile_limits['spacing_max'])+
#         '_angle_' + str(tile_limits['angle_min'])  +"_to_" + str(tile_limits['angle_max'])+
#         '_amp1_' + str(tile_limits['amp1_min'])  +"_to_" + str(tile_limits['amp1_max'])+
#         '_amp2_' + str(tile_limits['amp2_min'])  +"_to_" + str(tile_limits['amp2_max']))
        
    else:
        tile_limits = None
    

    
    if nofilteroverlaps==True:
        
        if wl == 488:
            wiener = 0.0001 # wiener = 0.0001 for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
            force_modamp = [0.8,0.1]  # [0.8,0.1] for 488nm,[0.8,0.25] for 532nm, [0.8,0.4] for 561nm
        elif wl == 532:
            wiener = 0.0003 # wiener = 0.0001 for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
            force_modamp = [1.0,0.2]  # [0.8,0.1] for 488nm,[0.8,0.25] for 532nm, [0.8,0.4] for 561nm
        elif wl == 560:
            wiener = 0.0002 # wiener = 0.0001 for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
            force_modamp = [1.0,0.2]  # [0.8,0.1] for 488nm,[0.8,0.25] for 532nm, [0.8,0.4] for 561nm
        elif wl == 642:
            wiener = 0.0002 # wiener = 0.0001 for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
            force_modamp = [1.0,0.2]  # [0.8,0.1] for 488nm,[0.8,0.25] for 532nm, [0.8,0.4] for 561nm
            
        user_text_process += '_NoFoverlaps_'
        user_text_process += 'Fmod_' + str(force_modamp)
        #user_text_process += 'forceotf_'
        if 'keeporder2' in  base_kwargs:
            del base_kwargs['keeporder2']
        base_kwargs.update(dict(nofilteroverlaps=nofilteroverlaps, forcemodamp = force_modamp))
    else:
        
        if wl == 488:
            wiener = 0.0005 #or 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
           
        elif wl == 532:
            wiener = 0.0015 #for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
            
        elif wl == 560:
            wiener = 0.002 #for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
            
        elif wl == 642:
            wiener = 0.002  #for 488nm, 0.00025 for 532nm,0.00025 for 561nm??? not sure??
           
            
        if 'forcemodamp' in  base_kwargs:
            del base_kwargs['forcemodamp']
        base_kwargs.update(dict(nofilteroverlaps=nofilteroverlaps))
        
    if keep_order2==True:
        user_text_process += '_keepod2_'
        #user_text_process += 'forceotf_'
        if 'otfamp' in  base_kwargs:
            del base_kwargs['otfamp']
        if 'nofilteroverlaps' in base_kwargs:
            del base_kwargs['nofilteroverlaps']
        base_kwargs.update(dict(keeporder2 = keep_order2))
        if 'forcemodamp' in  base_kwargs:
            del base_kwargs['forcemodamp']

    
    
  
    
  
    
    for otf in otfs:
                
        if "proc" not in os.path.split(raw)[1]: 

            ls = (wl/1000)/(2*0.775)
            

            # Test with actually measured OTF

            sim_kwargs = dict(
                input_file= raw,
                otf_file= otf,
                ls= ls,  # Lambda/2*Na, units: microns
                #ls = 0.315
                
            )
            
            
            OTF_filename = os.path.split(otf)[1].split('.')[0]
            base_kwargs.update(dict(nthreads=1, gammaApo=gammaApo, suppressR=suppressR, wiener=wiener))
            sim_kwargs.update(base_kwargs)
            print("\nSIMRECON BASE KEYWORD ARGUMENTS:" + str(base_kwargs))
            #user_text_process += '_w_' + str(wiener) + 'gApo_'+str(gammaApo)+'_supR_'+str(suppressR)
            if filter_tiles:
                user_text_process += '_tile_filter_'

            #create processed file output name
            sim_kwargs["output_file"] = sim_kwargs["input_file"].replace(".mrc",'_proc_' + OTF_filename + '_' +
                                    user_text_process + ".mrc")
            

            tile_process_name = sim_kwargs["input_file"].replace(".mrc",'_proc_' + OTF_filename + '_' +
             user_text_process +  '_tile' + str(tile_size) + "_pad" + str(tile_overlap) + ".mrc") #_tile16' + "_pad8" + ".mrc") 
            
            
            if len(tile_process_name) > 255:
                print(tile_process_name)
                print("""\nfilename-directory too long, > 256 characters. solutions to fix this:
            1) Rename your file, 2) copy and paste file into higher directory location with less parent directories""")
                break           


            if tile_process_name not in done_already:
                 

                    print('\nOTF FILENAME: ' +str(OTF_filename) + '\n')

                    print("SIM RECON DATA TO SAVE: " + str(tile_process_name) + "\n" + '\n' + '\n')
                    #print(tile_limits)
                    
                    %time sim_output = split_process_recombine(sim_kwargs["input_file"], tile_size, tile_overlap, sim_kwargs, tile_limits=tile_limits)
                    # HAD TO EDIT simrecon_utils.py lines 1488 and 1490 and comment out mrc.close() as it was failing.
                    
                    print(sim_output[0])
                    
                    with open(sim_output[0].replace(".mrc", ".txt"), "w") as myfile:
                        myfile.write(str(sim_kwargs))
                        myfile.write("\n" + "-" * 80 + "\n")
                        myfile.write("\n".join(sim_output[1]))
                    done_already.add(sim_output[0])
                    
           
            else:
                    print('DATASET ALREADY PROCESSED: ' + str(tile_process_name) + '\n' + '\n' + '\n')



# #

642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1.mrc
Processing 642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1.mrc (1024x1024) using Tiling recon is possible with Selected Tile: 128

SIMRECON BASE KEYWORD ARGUMENTS:{'nphases': 5, 'ndirs': 3, 'angle0': 1.29, 'negDangle': True, 'na': 0.85, 'nimm': 1.0, 'zoomfact': 2.0, 'background': 100.0, 'fastSIM': True, 'otfRA': True, 'dampenOrder0': True, 'k0searchall': True, 'equalizez': True, 'preciseapo': True, 'nofilteroverlaps': True, 'forcemodamp': [1.0, 0.2], 'nthreads': 1, 'gammaApo': 0.5, 'suppressR': 6.0, 'wiener': 0.0002}

OTF FILENAME: 642 20240611_125236_best

SIM RECON DATA TO SAVE: Z:\forJamesS\LID817_ROI4\LID817_ROI4_642\642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc_642 20240611_125236_best__NoFoverlaps_Fmod_[1.0, 0.2]_tile128_pad64.mrc





  0%|          | 0/8 [00:00<?, ?it/s]

Splitting and saving data:   0%|          | 0/64 [00:00<?, ?it/s]

[########################################] | 100% Completed | 43.67 s


Reading back processed:   0%|          | 0/64 [00:00<?, ?it/s]

Recombining:   0%|          | 0/64 [00:00<?, ?it/s]

MRC Raw data file is still open
MRC temp data file is still open
MRC Raw data file is now closed
MRC temp data file is now closed
CPU times: total: 1min 59s
Wall time: 1min 8s
Z:\forJamesS\LID817_ROI4\LID817_ROI4_642\642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc_642 20240611_125236_best__NoFoverlaps_Fmod_[1.0, 0.2]_tile128_pad64.mrc
642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1proc_OTF201909_19-20_best.mrc
642 nm 5 phases 0.81 NA React_All Linear SIM_cam1_1_proc_642 20240611_125236_best__NoFoverlaps_Fmod_[1.0, 0.2]_w_0.0002.mrc


###### 